# Install Base Packages

In [ ]:
!pip install git+https://github.com/boudinfl/pke.git
!pip install datasets
!python -m spacy download en_core_web_sm

  Cloning https://github.com/boudinfl/pke.git to /tmp/pip-req-build-g5hy956f
  Running command git clone --filter=blob:none --quiet https://github.com/boudinfl/pke.git /tmp/pip-req-build-g5hy956f
  Resolved https://github.com/boudinfl/pke.git to commit 69871ffdb720b83df23684fea53ec8776fd87e63
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 21.0 MB/s eta 0:00:00
  Created wheel for pke: filename=pke-2.0.0-py3-none-any.whl size=6160628 sha256=498ba97b70bb53e327683e2dbd101729f9816e62a4b94500c3cadb959cf6aab6
  Stored in directory: /tmp/pip-ephem-wheel-cache-he8eyudu/wheels/2e/78/39/76193c2a815f4cf34d67af2d338910453ecae3bde545185b65
Successfully built pke
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 150.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python 

# Restart Runtime

# Load and Preprocess Dataset

In [ ]:
from tqdm.notebook import tqdm
from datasets import load_dataset

benchmark = "Inspec"
dataset = load_dataset(
    "parquet",
    data_files={
        "train": "https://huggingface.co/datasets/taln-ls2n/inspec/resolve/refs/convert/parquet/raw/train/0000.parquet",
        "test":  "https://huggingface.co/datasets/taln-ls2n/inspec/resolve/refs/convert/parquet/raw/test/0000.parquet",
        "validation": "https://huggingface.co/datasets/taln-ls2n/inspec/resolve/refs/convert/parquet/raw/validation/0000.parquet"
    }
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


0000.parquet:   0%|          | 0.00/659k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/327k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/313k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

In [ ]:
def remove_empty_entries_from_dataset(data):
    for split in data.keys():
        original_count = len(data[split])

        data[split] = data[split].filter(
            lambda x: (
                x['id'] and
                x['title'] and
                x['abstract'] and
                x['keyphrases']
            )
        )

        cleaned_count = len(data[split])

        print(f"[{split}] Before data cleaning: {original_count}")
        print(f"[{split}] After data cleaning: {cleaned_count}")

    return data

dataset = remove_empty_entries_from_dataset(dataset)

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

[train] Before data cleaning: 1000
[train] After data cleaning: 1000


Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

[test] Before data cleaning: 500
[test] After data cleaning: 500


Filter:   0%|          | 0/500 [00:00<?, ? examples/s]

[validation] Before data cleaning: 500
[validation] After data cleaning: 500


In [ ]:
def combine_title_and_abstract(dataset):
    for split in dataset.keys():
        dataset[split] = dataset[split].map(
            lambda x: {"document": f"{x['title']}. {x['abstract']}"}
        )
    return dataset

dataset = combine_title_and_abstract(dataset)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'title', 'abstract', 'keyphrases', 'prmu', 'document'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['id', 'title', 'abstract', 'keyphrases', 'prmu', 'document'],
        num_rows: 500
    })
    validation: Dataset({
        features: ['id', 'title', 'abstract', 'keyphrases', 'prmu', 'document'],
        num_rows: 500
    })
})

# Define PKE Environment

**NOTE:** Unless otherwise noted in this notebook, the implementations are based on the [Python-based Keyphrase Extraction toolkit (PKE)](https://github.com/boudinfl/pke) by [Boudin (2016)](https://aclanthology.org/C16-2015/), tailored for our work.

In [ ]:
import spacy
from spacy.tokenizer import _get_regex_pattern

nlp = spacy.load("en_core_web_sm")

from spacy.lang.char_classes import ALPHA, ALPHA_LOWER, ALPHA_UPPER
from spacy.lang.char_classes import CONCAT_QUOTES, LIST_ELLIPSES, LIST_ICONS
from spacy.util import compile_infix_regex

infixes = (
    LIST_ELLIPSES
    + LIST_ICONS
    + [
        r"(?<=[0-9])[+\-\*^](?=[0-9-])",
        r"(?<=[{al}{q}])\.(?=[{au}{q}])".format(
            al=ALPHA_LOWER, au=ALPHA_UPPER, q=CONCAT_QUOTES
        ),
        r"(?<=[{a}]),(?=[{a}])".format(a=ALPHA),
        r"(?<=[{a}0-9])[:<>=/](?=[{a}])".format(a=ALPHA),
    ]
)

infix_re = compile_infix_regex(infixes)
nlp.tokenizer.infix_finditer = infix_re.finditer

In [ ]:
from nltk.stem.snowball import SnowballStemmer as Stemmer

all_data = []
references = []
stemmer = Stemmer('porter')

for split in dataset.keys():
    for sample in tqdm(dataset[split]):
        all_data.append(nlp(sample["document"]))

        sample_keyphrases = []
        for keyphrase in sample["keyphrases"]:
            tokens = [token.text for token in nlp(keyphrase)]
            stems = [stemmer.stem(tok.lower()) for tok in tokens]
            sample_keyphrases.append(" ".join(stems))
        references.append(sample_keyphrases)

  0%|          | 0/1000 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

  0%|          | 0/500 [00:00<?, ?it/s]

In [ ]:
from pke import compute_document_frequency
from string import punctuation
from pke import load_document_frequency_file

In [ ]:
compute_document_frequency(
    documents=all_data,
    output_file='data/{}.df.gz'.format(benchmark),
    language='en',
    normalization='stemming',
    stoplist=list(punctuation),
    n=5
)

df = load_document_frequency_file(input_file='data/{}.df.gz'.format(benchmark))

In [ ]:
from __future__ import absolute_import
from __future__ import division
from __future__ import print_function

import math
import logging

from pke.base import LoadFile

# Define AKE Methods

In [ ]:
# @title TF

class TF(LoadFile):

      def candidate_selection(self, n=3):
          self.ngram_selection(n=n)
          self.candidate_filtering()

      def candidate_weighting(self):
        for k, v in self.candidates.items():
            self.weights[k] = len(v.surface_forms)
            self.weights[k] += (self.candidates[k].offsets[0] * 1e-8)

In [ ]:
import networkx as nx

In [ ]:
# @title TextRank

class TextRank1(LoadFile):

    def __init__(self):
        super(TextRank1, self).__init__()
        self.graph = nx.Graph()

    def candidate_selection(self, pos=None):

        if pos is None:
            pos = {'NOUN', 'PROPN', 'ADJ'}

        self.longest_pos_sequence_selection(valid_pos=pos)

    def build_word_graph(self, window=2, pos=None):

        if pos is None:
            pos = {'NOUN', 'PROPN', 'ADJ'}

        text = [(word, sentence.pos[i] in pos) for sentence in self.sentences
                for i, word in enumerate(sentence.stems)]

        self.graph.add_nodes_from([word for word, valid in text if valid])

        for i, (node1, is_in_graph1) in enumerate(text):

            if not is_in_graph1:
                continue

            for j in range(i + 1, min(i + window, len(text))):
                node2, is_in_graph2 = text[j]
                if is_in_graph2 and node1 != node2:
                    self.graph.add_edge(node1, node2)

    def candidate_weighting(self,
                            window=2,
                            pos=None,
                            top_percent=None,
                            normalized=False):

        if pos is None:
            pos = {'NOUN', 'PROPN', 'ADJ'}

        self.build_word_graph(window=window, pos=pos)

        w = nx.pagerank(self.graph, alpha=0.85, tol=0.0001, weight=None)

        if top_percent is not None:

            nb_nodes = self.graph.number_of_nodes()
            to_keep = min(math.floor(nb_nodes * top_percent), nb_nodes)

            top_words = sorted(w, key=w.get, reverse=True)

            self.longest_keyword_sequence_selection(top_words[:int(to_keep)])

        for k in self.candidates.keys():
            tokens = self.candidates[k].lexical_form
            self.weights[k] = sum([w[t] for t in tokens])
            if normalized:
                self.weights[k] /= len(tokens)

            self.weights[k] += (self.candidates[k].offsets[0]*1e-8)

**NOTE:** The implementation of LMRank [(Giarelies and Karacapilidis, 2023)](https://ieeexplore.ieee.org/document/10179894) is based on [this](https://github.com/NC0DER/LMRank/blob/main/LMRank/model.py) and tailored to fit our work.

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 117.3 MB/s eta 0:00:00


In [ ]:
from __future__ import annotations
import torch
import numpy.typing
import faiss

from itertools import groupby
from operator import itemgetter
from difflib import get_close_matches
from sentence_transformers import SentenceTransformer
from typing import TypeVar, List, Tuple, Any

Model = TypeVar('Model')

In [ ]:
# @title LMRank

class LMRANK():
    def __init__(self: LMRANK) -> None:

        self.model = SentenceTransformer('all-mpnet-base-v2')
        self.text = None
        self.doc = None

    @staticmethod
    def remove_last_seps(string: str, seps: str = '!?.') -> str:
        sep_set = set(seps)
        for i in range(len(string) - 1, -1, -1):
            if string[i - 1] in sep_set:
                return string[:i - 1]
        return string

    @staticmethod
    def find_nth_occurence(string: str, substring: str, start: int, end: int, n: int) -> int:

        i = string.find(substring, start, end)
        while i >= 0 and n > 1:
            i = string.find(substring, i + len(substring))
            n -= 1
        return i

    @staticmethod
    def create_chunks(string: str, max_token_length: int, token_sep: str = ' ') -> List[str]:

        chunk_ranges = []
        chunk_start = 0
        chunk_end = 0

        while chunk_end < len(string):

            chunk_start = chunk_end

            next_sep_pos = LMRANK.find_nth_occurence(
                string, token_sep, chunk_start, len(string),
                max_token_length
            )

            if next_sep_pos == -1:
                chunk_end = len(string)
            else:
                chunk_end = next_sep_pos

            chunk_ranges.append((chunk_start, chunk_end))

        chunks = [string[i:j] for (i,j) in chunk_ranges]

        return chunks

    def extract_candidate_keyphrases(
            self: LMRANK, text: str, sentence_seps: str = '!?.',
            deduplicate: bool = True, keep_nouns_adjs: bool = True,
        ) -> List[Tuple[str, int]]:

        self.text = ' '.join(text.split())

        self.doc = nlp(self.text)

        candidate_keyphrases = [
            (LMRANK.remove_last_seps(chunk.text.lower(), sentence_seps), chunk.start)
            for chunk in self.doc.noun_chunks
            if chunk.text.lower() not in nlp.Defaults.stop_words
            and chunk[0].pos_ not in {'PRON', 'PART'}
            and all(
                term.pos_ in {'NOUN', 'ADJ'}
                if keep_nouns_adjs else True for term in chunk
            )
            and len(chunk.text) > 2
            and not chunk.text[:1].isdigit()
            and not any(term.like_url or term.like_email for term in chunk)
        ]

        candidate_keyphrases = {
            key: next(group)[1]
            for key, group in groupby(
                sorted(candidate_keyphrases, key = itemgetter(0)),
                itemgetter(0))
        }

        if deduplicate:
            string_similarity = 0.65

            for item in list(candidate_keyphrases):
                close_matches = get_close_matches(item, candidate_keyphrases.keys(),
                                                  cutoff = 0.65, n = 10)[1:]
                for close_match in close_matches:
                    if not item.count(' '):
                        candidate_keyphrases.pop(item, None)
                        break
                    elif (len(close_match) > len(item)
                            and len(get_close_matches(item, [close_match], n = 1, cutoff = string_similarity))):
                        candidate_keyphrases.pop(close_match, None)

        return list(candidate_keyphrases.items())

    def encode(
            self: LMRANK, string_list: List[str],
            multi_processing: bool = False, device: str = 'cuda'
        ) -> numpy.typing.NDArray[Any]:

        model = self.model

        if multi_processing:
            pool = model.start_multi_process_pool(target_devices = [device])
            embeddings = model.encode_multi_process(string_list, pool)
            model.stop_multi_process_pool(pool)
        else:
            embeddings = model.encode(string_list, device = device)
        return embeddings

    def model_token_length(self: LMRANK) -> int:

        model = self.model

        return model.max_seq_length

    def get_keyphrases_embeddings(
            self: LMRANK, candidate_keyphrases: List[Tuple[str, List[int]]]) -> numpy.typing.NDArray[Any]:

        embeddings = self.encode([keyphrase for keyphrase, _ in candidate_keyphrases])
        return embeddings

    def get_document_embedding(self: LMRANK) -> numpy.typing.NDArray[np.float32]:

        if len(self.doc) <= self.model_token_length():
            document_embedding = self.encode(self.text)
        else:
            chunks = LMRANK.create_chunks(self.text, self.model_token_length())
            document_embedding = np.mean(
                self.encode(chunks), axis = 0
            )

        return document_embedding

    def calculate_positional_scores(
            self: LMRANK, candidate_keyphrases: List[Tuple[str, int]]
        ) -> numpy.typing.NDArray[np.float32]:

        scores = np.array([
            1 / (position + 1)
            for _, position in candidate_keyphrases
        ])

        e_scores = numpy.exp(scores - np.max(scores))
        scores = e_scores / e_scores.sum(axis = 0)

        return scores

    def LMRank(
            self: LMRANK, text: str, sentence_seps: str = '.?!',
            deduplicate: bool = False, keep_nouns_adjs: bool = True, positional_feature: bool = False
        ) -> List[Tuple[str, float]]:

        candidate_keyphrases = self.extract_candidate_keyphrases(
            text, sentence_seps, deduplicate, keep_nouns_adjs
        )

        if not candidate_keyphrases:
            return []

        embeddings = self.get_keyphrases_embeddings(candidate_keyphrases)
        document_embedding = np.atleast_2d(self.get_document_embedding())

        unranked_ids = np.array(range(len(embeddings))).astype(np.int64)

        embedding_dim = len(embeddings[0])

        index = faiss.index_factory(embedding_dim, 'IDMap,Flat', faiss.METRIC_INNER_PRODUCT)

        faiss.normalize_L2(embeddings)

        index.add_with_ids(embeddings, unranked_ids)

        faiss.normalize_L2(document_embedding)

        similarities, ranked_ids = index.search(document_embedding, len(candidate_keyphrases))

        if positional_feature:
            scores = self.calculate_positional_scores(candidate_keyphrases)

            ranked_list = [
                (candidate_keyphrases[key_id][0], float(sim * score))
                for key_id, sim, score in zip(ranked_ids[0], similarities[0], scores)]

        else:
            ranked_list = [
                (candidate_keyphrases[key_id][0], float(sim))
                for key_id, sim in zip(ranked_ids[0], similarities[0])]

        seen_stems = {}
        for phrase, score in ranked_list:
            tokens = [token.text for token in nlp(phrase)]
            stems = [stemmer.stem(tok.lower()) for tok in tokens]
            stemmed_phrase = " ".join(stems)

            if stemmed_phrase not in seen_stems:
                seen_stems[stemmed_phrase] = score
            elif score > seen_stems[stemmed_phrase]:
                seen_stems[stemmed_phrase] = score

        ranked_list = sorted(seen_stems.items(), key=lambda x: x[1], reverse=True)

        return ranked_list

# Evaluate AKE Methods

In [ ]:
from pke.unsupervised import *
from timeit import default_timer as timer

In [ ]:
import numpy as np

In [ ]:
outputs = {}
outputs2 = {}
elapsed_times = {}
for model in [TF, TfIdf, KPMiner, YAKE, TextRank1, SingleRank, PositionRank, LMRANK]:
    outputs[model.__name__] = []
    outputs2[model.__name__] = []

    extractor = model()
    start = timer()

    if model.__name__ == "KPMiner":
        for i, doc in enumerate(tqdm(all_data)):
            extractor.load_document(input=doc, language='en')
            extractor.candidate_selection(lasf=3, cutoff=400)

            n_dynamic = len(references[i])

            extractor.candidate_weighting(df=df)

            outputs[model.__name__].append([u for u,v in extractor.get_n_best(n=n_dynamic, stemming=True)])
            doc_candidates = dict(extractor.weights)
            outputs2[model.__name__].append(doc_candidates)

    elif model.__name__ == "YAKE":
        for i, doc in enumerate(tqdm(all_data)):
            extractor.load_document(input=doc, language='en')
            extractor.candidate_selection(n=3)

            n_dynamic = len(references[i])

            use_stems = True
            extractor.candidate_weighting(use_stems=use_stems)

            outputs[model.__name__].append([u for u,v in extractor.get_n_best(n=n_dynamic, stemming=True)])
            doc_candidates = dict(extractor.weights)
            outputs2[model.__name__].append(doc_candidates)

    elif model.__name__ == "TextRank1":
        for i, doc in enumerate(tqdm(all_data)):
            extractor.load_document(input=doc, language='en')

            extractor.candidate_weighting(window=2, pos={'NOUN', 'PROPN', 'ADJ'}, top_percent=0.33)
            outputs[model.__name__].append([u for u,v in extractor.get_n_best(n=n_dynamic, stemming=True)])
            doc_candidates = dict(extractor.weights)
            outputs2[model.__name__].append(doc_candidates)

    elif model.__name__ == "SingleRank":
        for i, doc in enumerate(tqdm(all_data)):
            extractor.load_document(input=doc, language='en')
            extractor.candidate_selection(pos={'NOUN', 'PROPN', 'ADJ'})

            extractor.candidate_weighting(window=10, pos={'NOUN', 'PROPN', 'ADJ'})
            outputs[model.__name__].append([u for u,v in extractor.get_n_best(n=n_dynamic, stemming=True)])
            doc_candidates = dict(extractor.weights)
            outputs2[model.__name__].append(doc_candidates)

    elif model.__name__ == "PositionRank":
        for i, doc in enumerate(tqdm(all_data)):
            extractor.load_document(input=doc, language='en')
            extractor.candidate_selection(grammar="NP: {<ADJ>*<NOUN|PROPN>+}", maximum_word_number=3)

            extractor.candidate_weighting(window=10, pos={'NOUN', 'PROPN', 'ADJ'})
            outputs[model.__name__].append([u for u,v in extractor.get_n_best(n=n_dynamic, stemming=True)])
            doc_candidates = dict(extractor.weights)
            outputs2[model.__name__].append(doc_candidates)

    elif model.__name__ == "LMRANK":

        for i, text in enumerate(tqdm(all_data)):
            n_dynamic = len(references[i])

            text = str(text)
            ranked_list = extractor.LMRank(text=text)

            outputs[model.__name__].append([u for u, v in ranked_list[:n_dynamic]])
            outputs2[model.__name__].append({u: v for u, v in ranked_list})

    else:
        for i, doc in enumerate(tqdm(all_data)):
            extractor.load_document(input=doc, language='en')
            extractor.candidate_selection(n=3)

            n_dynamic = len(references[i])

            if model.__name__ == "TfIdf":
                extractor.candidate_weighting(df=df)
            else:
                extractor.candidate_weighting()
            outputs[model.__name__].append([u for u,v in extractor.get_n_best(n=n_dynamic, stemming=True)])
            doc_candidates = dict(extractor.weights)
            outputs2[model.__name__].append(doc_candidates)

    end = timer()
    elapsed_times[model.__name__] = end - start

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

  0%|          | 0/2000 [00:00<?, ?it/s]

In [ ]:
def evaluate_exact(top_N_candidates, references, cutoff=None):
    if cutoff is None:
        cutoff = len(references)
    cutoff = min(cutoff, len(top_N_candidates))
    P = len(set(top_N_candidates[:cutoff]) & set(references)) / len(top_N_candidates[:cutoff])
    R = len(set(top_N_candidates[:cutoff]) & set(references)) / len(references)
    F = (2 * P * R) / (P + R) if (P + R) > 0 else 0
    return (P, R, F)

def split_into_tokens(phrases):
    tokens = set()
    for phrase in phrases:
        tokens.update(phrase.split())
    return tokens

def evaluate_partial(top_N_candidates, references, cutoff=None):
    if cutoff is None:
        cutoff = len(references)
    cutoff = min(cutoff, len(top_N_candidates))

    predicted_tokens = split_into_tokens(top_N_candidates[:cutoff])
    reference_tokens = split_into_tokens(references)

    intersection = len(predicted_tokens & reference_tokens)

    P = intersection / len(predicted_tokens) if predicted_tokens else 0
    R = intersection / len(reference_tokens) if reference_tokens else 0
    F = (2 * P * R) / (P + R) if (P + R) > 0 else 0
    return (P, R, F)

def evaluate_harmonic(F1, pF1):
    return (2 * F1 * pF1) / (F1 + pF1) if (F1 + pF1) > 0 else 0

In [ ]:
print("## Benchmarking on {}".format(benchmark))
print("| Model       | it/s |     F    |   pF    |    hF    |")
print("| :---------- | ----:| -------: | ------: | -------: |")

results = []
full_hF1_dict = {}

for model in outputs:
    scores_exact = []
    scores_partial = []
    scores_harmonic = []

    for i, output in enumerate(outputs[model]):
        if not output:
            P_exact, R_exact, F_exact = (0, 0, 0)
            P_partial, R_partial, F_partial = (0, 0, 0)
            hF1 = 0
        else:
            P_exact, R_exact, F_exact = evaluate_exact(output, references[i], cutoff=None)
            P_partial, R_partial, F_partial = evaluate_partial(output, references[i], cutoff=None)
            hF1 = evaluate_harmonic(F_exact, F_partial)

        scores_exact.append((P_exact, R_exact, F_exact))
        scores_partial.append((P_partial, R_partial, F_partial))
        scores_harmonic.append(hF1)

    full_hF1_dict[model] = scores_harmonic

    P_exact, R_exact, F_exact = np.mean(scores_exact, axis=0)
    P_partial, R_partial, F_partial = np.mean(scores_partial, axis=0)
    hF1_mean = np.mean(scores_harmonic)

    print("| {}  | {:.5f} | {:.5f} | {:.5f} | {:.5f} |".format(
        model,
        len(all_data) / elapsed_times[model],
        F_exact,
        F_partial,
        hF1_mean
    ))

    results.append({
        "Model": model,
        "F": F_exact,
        "pF": F_partial,
        "hF": hF1_mean
    })

## Benchmarking on Inspec
| Model       | it/s |     F    |   pF    |    hF    |
| :---------- | ----:| -------: | ------: | -------: |
| TF  | 222.17350 | 0.09064 | 0.41419 | 0.12517 |
| TfIdf  | 229.60278 | 0.17844 | 0.46866 | 0.23447 |
| KPMiner  | 144.21215 | 0.07094 | 0.30462 | 0.09238 |
| YAKE  | 75.67535 | 0.19923 | 0.47966 | 0.26033 |
| TextRank1  | 232.74113 | 0.11608 | 0.36455 | 0.14896 |
| SingleRank  | 242.08964 | 0.23802 | 0.45005 | 0.28724 |
| PositionRank  | 198.22991 | 0.24411 | 0.42348 | 0.28694 |
| LMRANK  | 10.33691 | 0.30926 | 0.49582 | 0.36358 |


In [ ]:
import pandas as pd

In [ ]:
full_hF1 = pd.DataFrame(full_hF1_dict)

In [ ]:
df_results = pd.DataFrame(results)

**Save Results**

In [ ]:
import pickle

In [ ]:
with open("Inspec_full_hF1.pkl", "wb") as f:
    pickle.dump(full_hF1, f)

In [ ]:
with open("Inspec_results.pkl", "wb") as f:
    pickle.dump(df_results, f)

# Normalize Outputs

In [ ]:
import scipy.stats

In [ ]:
def normalize_outputs(outputs2):
    outputs3 = {}
    epsilon = 1e-8

    for method, documents in outputs2.items():
        normalized_documents = []

        for doc in documents:
            if not doc:
                normalized_documents.append({})
                continue

            keys = list(doc.keys())
            values = list(doc.values())

            ranks = scipy.stats.rankdata(values, method="dense")
            rank_dict = dict(zip(keys, ranks))

            x_min = min(ranks)
            x_max = max(ranks)

            if method in ["TF", "TfIdf", "KPMiner", "TextRank1", "SingleRank", "PositionRank", "LMRANK"]:
                norm_doc = {k: float((rank_dict[k] - x_min + epsilon) / (x_max - x_min + 2 * epsilon)) for k in doc}

            elif method in ["YAKE"]:
                norm_doc = {k: float((x_max - rank_dict[k] + epsilon) / (x_max - x_min + 2 * epsilon)) for k in doc}

            normalized_documents.append(norm_doc)

        outputs3[method] = normalized_documents

    return outputs3

In [ ]:
outputs3 = normalize_outputs(outputs2)

In [ ]:
with open("Inspec_outputs.pkl", "wb") as f:
    pickle.dump(outputs, f)

In [ ]:
with open("Inspec_outputs2.pkl", "wb") as f:
    pickle.dump(outputs2, f)

In [ ]:
with open("Inspec_outputs3.pkl", "wb") as f:
    pickle.dump(outputs3, f)

# Build DTMs

In [ ]:
from collections import defaultdict

In [ ]:
def build_DTM(outputs3, method, top_l=50):

    docs_scores = outputs3[method]
    score_sum = defaultdict(float)
    for doc_scores in docs_scores:
        for phrase, score in doc_scores.items():
            score_sum[phrase] += score
    top_phrases_ranked = sorted(score_sum.items(), key=lambda x: x[1], reverse=True)[:top_l]
    final_phrases = [phrase for phrase, _ in top_phrases_ranked]

    matrix = []
    for doc_scores in docs_scores:
        row = [doc_scores.get(term, 0) for term in final_phrases]
        matrix.append(row)

    df_matrix = pd.DataFrame(matrix, columns=final_phrases)
    return df_matrix

In [ ]:
def save_all_dtms(outputs3, dataset_name, method_list, top_l=50):
    dtm_dict = {}

    for method in method_list:
        df = build_DTM(outputs3, method, top_l=top_l)
        var_name = f"{dataset_name}_{method}"
        dtm_dict[var_name] = df

    with open(f"{dataset_name}_dtm_dict.pkl", "wb") as f:
        pickle.dump(dtm_dict, f)

    return dtm_dict

**Save Outputs**

In [ ]:
dataset_name = "Inspec"
method_list = ["TF", "TfIdf", "KPMiner", "YAKE", "TextRank1", "SingleRank", "PositionRank", "LMRANK"]
dtms = save_all_dtms(outputs3, dataset_name, method_list)

# Extract Requirements

In [ ]:
!pip freeze > AKE_for_Inspec.txt